In [0]:
"""
sim_hr_delta.py
===============
Simulation script: generates synthetic HR data and writes it into
Databricks Delta tables under  smart_odoo.silver

Tables covered (3 total):
    1. silver.hr_department   ← fixed 5 departments; seeded once, skipped on re-runs
    2. silver.hr_job          ← fixed 10 jobs;        seeded once, skipped on re-runs
    3. silver.hr_employee     ← 80 NEW employees every run

════════════════════════════════════════════════════════════
SIMULATION RULES
════════════════════════════════════════════════════════════

[GENERAL]
- Catalog / Schema   : smart_odoo.silver
- Source tag         : dwh_source_db = "python_sim"
- Date range         : 2023-01-01  →  2026-04-22  (wider range for HR context)
- All IDs            : MAX(existing_id) + 1  →  append-safe, zero duplicates
- company_id         : always 1  ("Smart Odoo LLC")

[DEPARTMENTS]  silver.hr_department  — fixed 5, seeded once
    1  IT          → Technology & Infrastructure
    2  Sales       → Revenue & Clients
    3  Finance     → Accounting & Money
    4  HR          → Human Resources
    5  Operations  → Logistics & Execution
- parent_id          : None for all top-level departments
- master_department_id: same as department (self-ref simplified)
- manager_id         : set after employees exist (None on first run,
                       updated by second run when employee pool is available)
- parent_path        : "/dept_id/"

[JOBS]  silver.hr_job  — fixed 10, seeded once
    1  IT Manager          → dept 1
    2  Backend Developer   → dept 1
    3  Data Analyst        → dept 1
    4  Sales Manager       → dept 2
    5  Sales Executive     → dept 2
    6  Accountant          → dept 3
    7  Finance Analyst     → dept 3
    8  HR Manager          → dept 4
    9  Recruiter           → dept 4
   10  Operations Officer  → dept 5
- contract_type_id   : 1=Full-Time | 2=Part-Time | 3=Contract (weighted 70/15/15)
- no_of_recruitment  : 1–5
- requirements       : nullable ~40 %

[EMPLOYEES]  silver.hr_employee  — 80 NEW rows every run
- name               : Egyptian first + last name combinations
- legal_name         : same as name (~80 %) or None
- work_email         : firstname.lastname@company.com
- work_phone         : +20 1X XXXXXXXX
- mobile_phone       : same as work_phone (~70 %) or different
- lang               : ar_EG (60 %) | en_US (40 %)
- certificate        : bachelor (50 %) | master (20 %) | doctor (5 %) | other (25 %)
- study_field        : relevant to department
- place_of_birth     : Egyptian city
- emergency_contact  : random name (nullable 30 %)
- birthday           : 1970-01-01 → 2000-12-31 range
- visa_expire        : nullable ~70 % (only for employees with permit_no)
- work_permit_expiration_date: nullable ~70 %
- active             : True (90 %) | False (10 %)
- birthday_public_display   : True ~60 %
- work_permit_scheduled_activity : False ~85 %
- parent_id (manager): references an existing employee (~70 % of employees)
- coach_id           : references an existing employee (~60 % of employees)
- work_contact_id    : same as hr_employee_id (~80 %) or None
- barcode            : EMP-NNNNNN
- today_location_name: Office | Remote | Field | None (weighted)

════════════════════════════════════════════════════════════
"""

import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    BooleanType, TimestampType
)

# ─────────────────────────────────────────────────────────────
# SPARK + CATALOG
# ─────────────────────────────────────────────────────────────
spark = SparkSession.builder.appName("sim_hr_delta").getOrCreate()

CATALOG    = "smart_odoo"
SCHEMA     = "silver"
SOURCE_DB  = "python_sim"
COMPANY_ID = 1
COMPANY    = "Smart Odoo LLC"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# ─────────────────────────────────────────────────────────────
# DATE HELPERS
# ─────────────────────────────────────────────────────────────
START_DATE    = datetime(2023, 1, 1)
END_DATE      = datetime(2026, 4, 22)
BIRTHDAY_START= datetime(1970, 1, 1)
BIRTHDAY_END  = datetime(2000, 12, 31)

def rand_date() -> datetime:
    return START_DATE + timedelta(
        days=random.randint(0, (END_DATE - START_DATE).days)
    )

def rand_birthday() -> datetime:
    return BIRTHDAY_START + timedelta(
        days=random.randint(0, (BIRTHDAY_END - BIRTHDAY_START).days)
    )

def maybe(val, prob_none: float = 0.3):
    return None if random.random() < prob_none else val

now = datetime.now()

# ─────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────
def existing_names(table: str, col: str = "name") -> set:
    try:
        return {r[col] for r in spark.sql(
            f"SELECT {col} FROM {CATALOG}.{SCHEMA}.{table}"
        ).collect()}
    except Exception:
        return set()

def max_id(table: str, col: str) -> int:
    try:
        return spark.sql(
            f"SELECT COALESCE(MAX({col}), 0) FROM {CATALOG}.{SCHEMA}.{table}"
        ).collect()[0][0]
    except Exception:
        return 0

def append_delta(df, table: str):
    (df.write.format("delta").mode("append")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}"))
    print(f"  ✅  {table}  (+{df.count()} rows)")


# ─────────────────────────────────────────────────────────────
# REFERENCE DATA
# ─────────────────────────────────────────────────────────────
DEPT_SEED = [
    (1, "IT",         "Technology & Infrastructure"),
    (2, "Sales",      "Revenue & Clients"),
    (3, "Finance",    "Accounting & Money"),
    (4, "HR",         "Human Resources"),
    (5, "Operations", "Logistics & Execution"),
]
DEPTS = {d[0]: d[1] for d in DEPT_SEED}

JOB_SEED = [
    (1,  "IT Manager",          1, "Lead & oversee IT infrastructure"),
    (2,  "Backend Developer",   1, "Build server-side applications"),
    (3,  "Data Analyst",        1, "Analyze data and generate insights"),
    (4,  "Sales Manager",       2, "Manage sales team and targets"),
    (5,  "Sales Executive",     2, "Drive client acquisition"),
    (6,  "Accountant",          3, "Manage financial records"),
    (7,  "Finance Analyst",     3, "Analyze financial performance"),
    (8,  "HR Manager",          4, "Oversee HR operations"),
    (9,  "Recruiter",           4, "Source and hire talent"),
    (10, "Operations Officer",  5, "Coordinate logistics and execution"),
]
JOBS      = {j[0]: (j[1], j[2]) for j in JOB_SEED}   # id → (name, dept_id)
DEPT_JOBS = {}                                          # dept_id → [job_ids]
for jid, jname, did, _ in JOB_SEED:
    DEPT_JOBS.setdefault(did, []).append(jid)

CONTRACT_TYPES = {1: "Full-Time", 2: "Part-Time", 3: "Contract"}
CT_POOL = [1]*70 + [2]*15 + [3]*15

STUDY_FIELDS = {
    1: ["Computer Science", "Information Technology", "Software Engineering", "Data Science"],
    2: ["Business Administration", "Marketing", "Economics"],
    3: ["Accounting", "Finance", "Economics", "Business Administration"],
    4: ["Human Resources", "Psychology", "Business Administration"],
    5: ["Logistics", "Supply Chain", "Operations Management"],
}

FIRST_NAMES = [
    "Ahmed", "Omar", "Mohamed", "Ali", "Youssef", "Kareem", "Hassan",
    "Ibrahim", "Mahmoud", "Khaled", "Tamer", "Walid", "Hossam", "Sherif",
    "Nour", "Sara", "Mariam", "Aya", "Dina", "Rania", "Hana", "Yasmin",
    "Mona", "Amira", "Fatma", "Noha", "Samar",
]
LAST_NAMES = [
    "Hassan", "Ali", "Mohamed", "Said", "Younis", "Fathy", "Nabil",
    "Mahmoud", "Saleh", "Mostafa", "Abdallah", "Ramadan", "Khalil",
    "Mansour", "Ragab", "Gaber", "Zaki", "Fahmy", "Shawky", "Badawi",
]

CITIES = [
    "Cairo", "Alexandria", "Giza", "Mansoura", "Tanta",
    "Aswan", "Luxor", "Suez", "Port Said", "Ismailia",
]

LOCATIONS = ["Office"] * 50 + ["Remote"] * 30 + ["Field"] * 15 + [None] * 5
LANGS     = ["ar_EG"] * 60 + ["en_US"] * 40
CERTS     = ["bachelor"] * 50 + ["master"] * 20 + ["doctor"] * 5 + ["other"] * 25
SCHOOLS   = [
    "Cairo University", "Ain Shams University", "Alexandria University",
    "American University in Cairo", "Helwan University",
    "Mansoura University", "Zagazig University",
]

def make_email(name: str) -> str:
    return name.lower().replace(" ", ".") + "@company.com"

def make_phone() -> str:
    return f"+20 1{random.randint(0,2)}{random.randint(10000000, 99999999)}"


# ═══════════════════════════════════════════════════════════════
# 1. HR_DEPARTMENT  (5 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
dept_schema = StructType([
    StructField("hr_department_id",         IntegerType(),   True),
    StructField("company_id",               IntegerType(),   True),
    StructField("company_name",             StringType(),    True),
    StructField("parent_id",                IntegerType(),   True),
    StructField("parent_name",              StringType(),    True),
    StructField("manager_id",               IntegerType(),   True),
    StructField("manager_name",             StringType(),    True),
    StructField("master_department_id",     IntegerType(),   True),
    StructField("master_department_name",   StringType(),    True),
    StructField("name",                     StringType(),    True),
    StructField("parent_path",              StringType(),    True),
    StructField("note",                     StringType(),    True),
    StructField("active",                   BooleanType(),   True),
    StructField("created_at",               TimestampType(), True),
    StructField("updated_at",               TimestampType(), True),
    StructField("dwh_loaded_at",            TimestampType(), True),
    StructField("dwh_source_db",            StringType(),    True),
])

existing_dept_names = existing_names("hr_department")
dept_rows = []
for did, dname, dnote in DEPT_SEED:
    if dname in existing_dept_names:
        continue
    dept_rows.append(Row(
        hr_department_id       = did,
        company_id             = COMPANY_ID,
        company_name           = COMPANY,
        parent_id              = None,
        parent_name            = None,
        manager_id             = None,         # set after employees exist
        manager_name           = None,
        master_department_id   = did,
        master_department_name = dname,
        name                   = dname,
        parent_path            = f"/{did}/",
        note                   = dnote,
        active                 = True,
        created_at             = now,
        updated_at             = now,
        dwh_loaded_at          = now,
        dwh_source_db          = SOURCE_DB,
    ))

if dept_rows:
    append_delta(spark.createDataFrame(dept_rows, schema=dept_schema), "hr_department")
else:
    print("  ⏭   hr_department  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 2. HR_JOB  (10 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
job_schema = StructType([
    StructField("hr_job_id",           IntegerType(),   True),
    StructField("company_id",          IntegerType(),   True),
    StructField("company_name",        StringType(),    True),
    StructField("department_id",       IntegerType(),   True),
    StructField("department_name",     StringType(),    True),
    StructField("user_id",             IntegerType(),   True),
    StructField("user_name",           StringType(),    True),
    StructField("contract_type_id",    IntegerType(),   True),
    StructField("contract_type_name",  StringType(),    True),
    StructField("sequence",            IntegerType(),   True),
    StructField("no_of_recruitment",   IntegerType(),   True),
    StructField("name",                StringType(),    True),
    StructField("description",         StringType(),    True),
    StructField("requirements",        StringType(),    True),
    StructField("active",              BooleanType(),   True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("dwh_loaded_at",       TimestampType(), True),
    StructField("dwh_source_db",       StringType(),    True),
])

existing_job_names = existing_names("hr_job")
job_rows = []
for jid, jname, jdept, jdesc in JOB_SEED:
    if jname in existing_job_names:
        continue
    ct_id = random.choice(CT_POOL)
    uid   = random.randint(1, 10)
    job_rows.append(Row(
        hr_job_id          = jid,
        company_id         = COMPANY_ID,
        company_name       = COMPANY,
        department_id      = jdept,
        department_name    = DEPTS[jdept],
        user_id            = uid,
        user_name          = f"User_{uid}",
        contract_type_id   = ct_id,
        contract_type_name = CONTRACT_TYPES[ct_id],
        sequence           = jid,
        no_of_recruitment  = random.randint(1, 5),
        name               = jname,
        description        = jdesc,
        requirements       = maybe(f"Min 2 years experience in {jname}", 0.4),
        active             = True,
        created_at         = now,
        updated_at         = now,
        dwh_loaded_at      = now,
        dwh_source_db      = SOURCE_DB,
    ))

if job_rows:
    append_delta(spark.createDataFrame(job_rows, schema=job_schema), "hr_job")
else:
    print("  ⏭   hr_job  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 3. HR_EMPLOYEE  — 80 NEW rows every run
# ═══════════════════════════════════════════════════════════════

emp_id_start = max_id("hr_employee", "hr_employee_id") + 1
print(f"\nStarting hr_employee_id from : {emp_id_start}")

# Load existing employee IDs for parent / coach references
try:
    existing_emp_ids = [
        r.hr_employee_id for r in
        spark.sql(f"SELECT hr_employee_id FROM {CATALOG}.{SCHEMA}.hr_employee").collect()
    ]
except Exception:
    existing_emp_ids = []

emp_schema = StructType([
    StructField("hr_employee_id",               IntegerType(),   True),
    StructField("company_id",                   IntegerType(),   True),
    StructField("company_name",                 StringType(),    True),
    StructField("user_id",                      IntegerType(),   True),
    StructField("user_name",                    StringType(),    True),
    StructField("parent_id",                    IntegerType(),   True),
    StructField("parent_name",                  StringType(),    True),
    StructField("coach_id",                     IntegerType(),   True),
    StructField("coach_name",                   StringType(),    True),
    StructField("work_contact_id",              IntegerType(),   True),
    StructField("work_contact_name",            StringType(),    True),
    StructField("name",                         StringType(),    True),
    StructField("legal_name",                   StringType(),    True),
    StructField("work_phone",                   StringType(),    True),
    StructField("mobile_phone",                 StringType(),    True),
    StructField("work_email",                   StringType(),    True),
    StructField("lang",                         StringType(),    True),
    StructField("certificate",                  StringType(),    True),
    StructField("study_field",                  StringType(),    True),
    StructField("study_school",                 StringType(),    True),
    StructField("place_of_birth",               StringType(),    True),
    StructField("emergency_contact",            StringType(),    True),
    StructField("emergency_phone",              StringType(),    True),
    StructField("permit_no",                    StringType(),    True),
    StructField("visa_no",                      StringType(),    True),
    StructField("barcode",                      StringType(),    True),
    StructField("today_location_name",          StringType(),    True),
    StructField("salary_distribution",          StringType(),    True),
    StructField("employee_properties",          StringType(),    True),
    StructField("active",                       BooleanType(),   True),
    StructField("birthday_public_display",      BooleanType(),   True),
    StructField("work_permit_scheduled_activity", BooleanType(), True),
    StructField("birthday",                     TimestampType(), True),
    StructField("visa_expire",                  TimestampType(), True),
    StructField("work_permit_expiration_date",  TimestampType(), True),
    StructField("created_at",                   TimestampType(), True),
    StructField("updated_at",                   TimestampType(), True),
    StructField("dwh_loaded_at",                TimestampType(), True),
    StructField("dwh_source_db",                StringType(),    True),
])

emp_rows   = []
emp_id     = emp_id_start
this_run_emps: list[tuple[int, str]] = []   # (id, name) built this run for intra-run refs

for _ in range(80):

    # pick department & matching job
    dept_id   = random.choice(list(DEPTS.keys()))
    dept_name = DEPTS[dept_id]
    job_id    = random.choice(DEPT_JOBS[dept_id])
    job_name  = JOBS[job_id][0]

    # name & contact
    full_name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
    email     = make_email(full_name)
    phone     = make_phone()
    mobile    = phone if random.random() < 0.7 else make_phone()

    # manager / coach — prefer existing employees, then same-run pool
    ref_pool  = existing_emp_ids + [e[0] for e in this_run_emps]
    name_pool = {e[0]: e[1] for e in this_run_emps}

    if ref_pool and random.random() < 0.7:
        parent_id   = random.choice(ref_pool)
        parent_name = name_pool.get(parent_id, f"Employee_{parent_id}")
    else:
        parent_id, parent_name = None, None

    if ref_pool and random.random() < 0.6:
        coach_id   = random.choice(ref_pool)
        coach_name = name_pool.get(coach_id, f"Employee_{coach_id}")
    else:
        coach_id, coach_name = None, None

    # work contact
    wc_id   = emp_id if random.random() < 0.8 else None
    wc_name = full_name if wc_id else None

    # education
    cert        = random.choice(CERTS)
    study_field = random.choice(STUDY_FIELDS.get(dept_id, ["General Studies"]))
    school      = random.choice(SCHOOLS)

    # permit / visa (only some employees)
    has_permit  = random.random() < 0.25
    permit_no   = f"PRM-{random.randint(100000,999999)}" if has_permit else None
    visa_no     = f"VIS-{random.randint(100000,999999)}" if has_permit else None
    visa_exp    = (now + timedelta(days=random.randint(30, 730))) if has_permit and random.random() > 0.3 else None
    wp_exp      = (now + timedelta(days=random.randint(30, 730))) if has_permit and random.random() > 0.3 else None
    wp_sched    = has_permit and random.random() < 0.15

    # emergency contact
    ec_name  = maybe(f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}", 0.3)
    ec_phone = make_phone() if ec_name else None

    # hire date ~ work date
    hire_date = rand_date()

    emp_rows.append(Row(
        hr_employee_id                  = emp_id,
        company_id                      = COMPANY_ID,
        company_name                    = COMPANY,
        user_id                         = emp_id,
        user_name                       = full_name,
        parent_id                       = parent_id,
        parent_name                     = parent_name,
        coach_id                        = coach_id,
        coach_name                      = coach_name,
        work_contact_id                 = wc_id,
        work_contact_name               = wc_name,
        name                            = full_name,
        legal_name                      = full_name if random.random() < 0.8 else None,
        work_phone                      = phone,
        mobile_phone                    = mobile,
        work_email                      = email,
        lang                            = random.choice(LANGS),
        certificate                     = cert,
        study_field                     = study_field,
        study_school                    = school,
        place_of_birth                  = random.choice(CITIES),
        emergency_contact               = ec_name,
        emergency_phone                 = ec_phone,
        permit_no                       = permit_no,
        visa_no                         = visa_no,
        barcode                         = f"EMP-{emp_id:06d}",
        today_location_name             = random.choice(LOCATIONS),
        salary_distribution             = None,
        employee_properties             = None,
        active                          = random.random() < 0.9,
        birthday_public_display         = random.random() < 0.6,
        work_permit_scheduled_activity  = wp_sched,
        birthday                        = rand_birthday(),
        visa_expire                     = visa_exp,
        work_permit_expiration_date     = wp_exp,
        created_at                      = hire_date,
        updated_at                      = hire_date,
        dwh_loaded_at                   = now,
        dwh_source_db                   = SOURCE_DB,
    ))

    this_run_emps.append((emp_id, full_name))
    existing_emp_ids.append(emp_id)   # make available for next iterations
    emp_id += 1


df_emp = spark.createDataFrame(emp_rows, schema=emp_schema)
append_delta(df_emp, "hr_employee")

print("\n✅  DONE — All 3 HR Delta tables written to smart_odoo.silver")